# VLM eval inspection — DroneWaste + AerialWaste

Review what the trained VLM actually **said** per image, next to the ground-truth
labels, for any run under `results/vlm_eval/`.

The headline F1 hides the thing that matters most here: the 150K-SFT checkpoint
sprayed nearly every label at every image, and the 819K one answers `none` on
~70-79% of images. Same F1 range, opposite failure. This notebook is for looking
at that directly, and for the A/B (`COMPARE`) of the two checkpoints on the same
image.

Reads `test_eval.json` + `raw_responses.jsonl`; images come from
`$WASTE_DATA_ROOT`. Kernel: **myenv** (the only env here with ipykernel; needs just matplotlib + pillow, no torch).


In [1]:
import json, os, textwrap, random
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image

EVAL_ROOT = Path("/leonardo_scratch/large/userexternal/adiecidu/waste_vlm/results/vlm_eval")
DATA_ROOT = Path(os.environ.get(
    "WASTE_DATA_ROOT",
    "/leonardo_scratch/large/userexternal/adiecidu/waste_vlm/data"))
IMAGE_DIR = {
    "dw_paper10": DATA_ROOT / "dronewaste" / "images",
    "aw_m2":      DATA_ROOT / "aerialwaste" / "images",
    "aw_m4":      DATA_ROOT / "aerialwaste" / "images",
}


def load_run(name):
    """name = dir name under results/vlm_eval (or a full path)."""
    d = Path(name) if Path(name).is_absolute() else EVAL_ROOT / name
    report = json.loads((d / "test_eval.json").read_text())
    records = [json.loads(l) for l in
               (d / "raw_responses.jsonl").read_text().splitlines() if l.strip()]
    return report, records


def outcome(rec):
    """Per-image outcome — the axis the aggregate metrics average away."""
    gt, pred = set(rec.get("gt", [])), set(rec.get("parsed", []))
    if not gt and not pred:   return "tn"       # correctly silent
    if gt and pred == gt:     return "tp"       # exact hit
    if gt and not pred:       return "fn"       # silent on real waste
    if pred and not gt:       return "fp"       # invented waste
    return "partial"                            # overlapping label sets


def runs_table():
    rows = []
    for d in sorted(EVAL_ROOT.glob("vlm_*")):
        f = d / "test_eval.json"
        if not f.exists():
            continue
        r = json.loads(f.read_text())
        lpi = r.get("labels_per_image", {})
        rows.append((d.name, r["dataset"], r["prompt_style"], r["n_test"],
                     r["micro"]["f1"], r["micro"]["precision"], r["micro"]["recall"],
                     r.get("n_empty_parse", -1), lpi.get("pred_mean", float("nan"))))
    print(f"{'run':<62}{'dataset':<12}{'prompt':<14}{'n':>5}{'micF1':>7}"
          f"{'P':>7}{'R':>7}{'empty':>7}{'lab/img':>8}")
    for x in rows:
        print(f"{x[0]:<62}{x[1]:<12}{x[2]:<14}{x[3]:>5}{x[4]:>7.3f}"
              f"{x[5]:>7.3f}{x[6]:>7.3f}{x[7]:>7}{x[8]:>8.2f}")
    return rows


_ = runs_table()

run                                                           dataset     prompt            n  micF1      P      R  empty lab/img
vlm_cradiov4-so_r768ps2_dw_paper10_closed_vocab               dw_paper10  closed_vocab   1504  0.060  0.031  0.855      0     nan
vlm_cradiov4-so_r768ps2_dw_paper10_open_cot                   dw_paper10  open_cot       1504  0.106  0.065  0.285    402     nan
vlm_cradiov4-so_r768ps2_dw_paper10_open_cot_confident         dw_paper10  open_cot_confident 1504  0.084  0.053  0.201    610     nan
vlm_cradiov4-so_r768ps2_finetune_aw_m2_closed_vocab           aw_m2       closed_vocab    581  0.289  0.417  0.222    505    0.43
vlm_cradiov4-so_r768ps2_finetune_aw_m4_closed_vocab           aw_m4       closed_vocab    581  0.166  0.091  0.905     24    5.23
vlm_cradiov4-so_r768ps2_finetune_next_aw_m2_closed_vocab      aw_m2       closed_vocab    581  0.097  0.490  0.054    530    0.09
vlm_cradiov4-so_r768ps2_finetune_next_aw_m2_open_cot          aw_m2       open_cot    

## Pick a run

`RUN` is the run you are inspecting. Set `COMPARE` to a second run to see both
checkpoints' answers on the same image (that is the 150K vs 819K ablation), or
`None` for a single run.

`SELECT` filters by outcome — `fn` is the interesting one for the 819K model
(waste is there, it said nothing), `fp` for the 150K one.

In [ ]:
# ---- choose what to inspect (edit, then re-run this cell + the viewer) ----
RUN     = "vlm_cradiov4-so_r768ps2_finetune_next_aw_m4_closed_vocab"
COMPARE = None          # e.g. "vlm_cradiov4-so_r768ps2_dw_paper10_closed_vocab" (150K, same images)
SELECT  = "mixed"       # mixed | tp | fn | fp | tn | partial
N       = 12            # how many images to show
SEED    = 0
ONLY_GT_CLASS = None    # e.g. "Tyres" — restrict to images whose GT contains this class

report, records = load_run(RUN)
img_dir = IMAGE_DIR[report["dataset"]]
cmp_report, cmp_by_file = None, {}
if COMPARE:
    cmp_report, cmp_records = load_run(COMPARE)
    cmp_by_file = {r["file"]: r for r in cmp_records}

print(f"{RUN}\n  dataset={report['dataset']} prompt={report['prompt_style']} "
      f"n={report['n_test']} microF1={report['micro']['f1']:.3f} "
      f"P={report['micro']['precision']:.3f} R={report['micro']['recall']:.3f}")
if "labels_per_image" in report:
    lpi, bp = report["labels_per_image"], report["binary_presence"]
    print(f"  labels/img pred={lpi['pred_mean']:.2f} gt={lpi['gt_mean']:.2f} | "
          f"binary waste/no-waste F1={bp['f1']:.3f} "
          f"P={bp['precision']:.3f} R={bp['recall']:.3f}")
print("  outcomes:", dict(Counter(outcome(r) for r in records).most_common()))
if cmp_report:
    print(f"{COMPARE}\n  microF1={cmp_report['micro']['f1']:.3f}  "
          f"outcomes: {dict(Counter(outcome(r) for r in cmp_records).most_common())}")

pool = records if SELECT == "mixed" else [r for r in records if outcome(r) == SELECT]
if ONLY_GT_CLASS:
    pool = [r for r in pool if ONLY_GT_CLASS in r.get("gt", [])]
random.Random(SEED).shuffle(pool)
picked = pool[:N]
print(f"\n{len(pool)} records match SELECT={SELECT}"
      + (f" / GT contains {ONLY_GT_CLASS!r}" if ONLY_GT_CLASS else "")
      + f" — showing {len(picked)}")

## Viewer

In [ ]:
def show_answer(rec, tag):
    gt, pred = set(rec.get("gt", [])), set(rec.get("parsed", []))
    print(f"  [{tag}] {outcome(rec).upper()}")
    if "raw_turn1" in rec:
        print(textwrap.fill(f"describe: {rec['raw_turn1']}", 100,
                            initial_indent="    ", subsequent_indent="              "))
    print(textwrap.fill(f"answer  : {rec.get('raw', '')}", 100,
                        initial_indent="    ", subsequent_indent="              "))
    print(f"    parsed  : {sorted(pred) or '—'}")
    miss, extra = sorted(gt - pred), sorted(pred - gt)
    if miss or extra:
        bits = ([f"missed {miss}"] if miss else []) + ([f"extra {extra}"] if extra else [])
        print("    delta   : " + "; ".join(bits))


for rec in picked:
    path = img_dir / rec["file"]
    if path.exists():
        fig, ax = plt.subplots(figsize=(5.5, 5.5))
        ax.imshow(Image.open(path).convert("RGB"))
        ax.set_title(rec["file"], fontsize=9)
        ax.axis("off")
        plt.show()
    else:
        print(f"[image not on disk: {path}]")
    print(f"  GT      : {sorted(rec.get('gt', [])) or '—'}")
    show_answer(rec, "this run")
    if COMPARE:
        other = cmp_by_file.get(rec["file"])
        if other:
            show_answer(other, "compare")
        else:
            print("  [compare] image not in the other run")
    print("-" * 100)

## Aggregates

What the model says across the whole run, without sampling. The answer histogram
is usually the fastest way to see a degenerate mode (one string dominating), and
the per-class table shows whether a class is never predicted at all vs predicted
and wrong.

In [ ]:
print("most common raw answers")
for txt, k in Counter(r.get("raw", "").strip()[:90] for r in records).most_common(15):
    print(f"  {k:>5}  {txt!r}")

print("\nlabels asserted per image")
for k, c in sorted(Counter(len(r.get("parsed", [])) for r in records).items()):
    print(f"  {k} label(s): {c:>5}  {'#' * int(60 * c / len(records))}")

print("\nper class: gt support / times predicted / exact co-occurrence")
cats = list(report["per_class"].keys())
for c in cats:
    sup  = sum(1 for r in records if c in r.get("gt", []))
    pred = sum(1 for r in records if c in r.get("parsed", []))
    hit  = sum(1 for r in records if c in r.get("gt", []) and c in r.get("parsed", []))
    f1   = report["per_class"][c].get("f1")
    print(f"  {c[:44]:<46} gt={sup:>4}  pred={pred:>4}  hit={hit:>4}  "
          f"F1={'n/a' if f1 is None else f'{f1:.3f}'}")

## Notes

- Outcome colours/labels: `tp` exact hit · `partial` overlapping labels ·
  `fn` silent on real waste · `fp` invented waste · `tn` correctly silent.
- `parsed` is what the eval's parser extracted from `answer`. A non-empty answer
  with empty `parsed` means the model named something outside the label set
  (e.g. it says "wood" on DroneWaste, whose closest class is `Pallets`) — worth
  watching before concluding the model missed the object.
- open_cot runs carry `describe` (turn 1) as well; on the 819K checkpoint those
  come back terse and generic, which is why turn 2 has nothing to work with.
- Headless equivalent (writes a self-contained HTML gallery with thumbnails):
  `python scripts/inspect_outputs.py --run <eval_dir> --out page.html [--compare <dir>]`
